AI

In [162]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

task 1

In [163]:
def robust_file_reader(f_path: Path | str) -> tuple[str, str]: 
    """Gets a path to a file where it checks if it's UTF-8 or Latin-1 and saves data to a tuple."""
    # Path(Path()) fungerer som Path() så det går bra hvis de skriver path 
    f_path = Path(f_path) 
    byte = f_path.read_bytes()  
    
    try: # Setter dataen til variabel
        file_text_content = byte.decode("utf8") 
        encoding_used = "UTF-8"
    except UnicodeDecodeError: # Hvis den får en feil så setter den til hva det andre valge kunne være
        file_text_content = byte.decode("latin-1")
        encoding_used = "Latin-1"

    return file_text_content, encoding_used


file_text, encoding = robust_file_reader("data/2026/library_cafe/POS_01/register_sales.log")



def log_passer(f_read: str) -> list[dict]:
    """Makes a list of dictonary with valid sales line"""
    valid_list = []

    for line in f_read.splitlines():
        if line.startswith("=") or line.startswith("-") or line.startswith("[DEBUG]"):
            continue

        timestamp_match = re.search(r"\[(\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\]\s", line)
        register_id_match = re.search(r"\[([A-Z]{3}_\d{2})\]", line)
        item_match = re.search(r"item=([a-zA-Z]+)\s", line)
        price_match = re.search(r"price=(\d+\.\d+)\s", line)
        currency_match = re.search(r"\s([A-Z]{3})\s", line)
        status_match = re.search(r"\(([A-Z]+)\)", line)

        if timestamp_match and register_id_match and item_match and price_match and currency_match and status_match:
            match_dict = {
                "timestamp": timestamp_match.group(1),
                "register_id": register_id_match.group(1), 
                "item": item_match.group(1),
                "price": float(price_match.group(1)), 
                "currency": currency_match.group(1),
                "status": status_match.group(1)
            }

            valid_list.append(match_dict)
        
    return valid_list


task 2

In [164]:
def recursive_directory_search(path_search: str | Path) -> list[Path]:
    """Checks all the branches under the dictonary it gets"""
    path_search = Path(path_search)
    path_list = []

    for item in path_search.iterdir(): # sjekker igjennom directory og alle filene eller directory inne i item
        if item.is_dir():
            path_list.extend(recursive_directory_search(item))
        elif item.is_file() and item.suffix == ".log": # suffix er det samme som .endswith bare for objekter
            path_list.append(item)
            
    return path_list

print(recursive_directory_search(Path("data")))

file_text, encoding = robust_file_reader(Path("data/store_config.json"))
info = json.loads(file_text)


def register_travel_recursive(value: list | dict) -> set[str]:
    """Checks if it got a list or dictonary and cheks if it have id's that is active and returns it"""
    set_id = set()

    if isinstance(value, dict):
        if value.get("active") is True: 
            set_id.add(value["id"])
        for inside in value.values():
            set_id.update(register_travel_recursive(inside))
    elif isinstance(value, list):
        for inside in value:
            set_id.update(register_travel_recursive(inside))

    return set_id

print(register_travel_recursive(info)) # ta bort



[WindowsPath('data/2026/library_cafe/POS_01/register_sales.log'), WindowsPath('data/2026/library_cafe/POS_02/register_sales.log'), WindowsPath('data/2026/library_cafe/POS_03/register_sales.log'), WindowsPath('data/2026/science_building/POS_01/register_sales.log'), WindowsPath('data/2026/science_building/POS_02/register_sales.log'), WindowsPath('data/2026/science_building/POS_03/register_sales.log'), WindowsPath('data/2026/student_union/POS_01/register_sales.log'), WindowsPath('data/2026/student_union/POS_02/register_sales.log'), WindowsPath('data/2026/student_union/POS_03/register_sales.log')]
{'POS_02', 'POS_01', 'POS_03'}


**task 3** <br>
Functions for cleaning and analysing data using NumPy

In [165]:
def remove_price_outliers(price_data: list[float] | np.ndarray) -> np.ndarray:
    """
    Removes hardware glitch outliers from price data.

    Parameters:
        price_data : array-like
            Input price values.

    Returns:
        np.ndarray
            Cleaned 1D float array.
    """

    #Converts to 1D NumPy float array
    prices = np.asarray(price_data, dtype=float).flatten()

    # Calculate mean and std avvik
    mean = np.mean(prices)
    std = np.std(prices)

    #identify outliers
    outliers = np.abs(prices - mean) > 3 * std

    #find median of valid prices
    valid_prices = prices[~outliers]
    median_value = np.median(valid_prices)

    #replace outliers with median
    prices[outliers] = median_value

    return prices

def compute_refund_ratio(
        sales_count: np.ndarray,
        refund_count: np.ndarray
) -> np.ndarray:
    """ 
    Computes element-wise refund ratios.

    Refund Ratio = Refund Count / (Sales Count + 1e-6)

    Parameters:
        sales_count : np.ndarray
        refund_count : np.ndarray

    Returns:
        np.ndarray
            Array of refund ratios.
    """

    epsilon = 1e-6

    #accept both lists and nupmpy arrays
    sales = np.asarray(sales_count, dtype=float)
    refunds = np.asarray(refund_count, dtype=float)
    
    return refund_count / (sales_count + epsilon)

In [166]:
#testcelle
test = np.array([4.0] * 20 + [999.0])
cleaned = remove_price_outliers(test)
assert 999.0 not in cleaned
assert test[-1] == 999.0          # the original array is unchanged
print("Outlier test: OK!")

ratio = compute_refund_ratio(np.array([10, 0]), np.array([2, 0]))
assert np.isclose(ratio[0], 0.2) and ratio[1] == 0.0   # no division by zero
print("Refund ratio test: OK!")

Outlier test: OK!
Refund ratio test: OK!


task 5

In [167]:
all_records = recursive_directory_search("data/2026")
active_ids = register_travel_recursive(info)


all_records_list = []

for file in all_records:
   file_text, encoding = robust_file_reader(file)
   valid_text = log_passer(file_text)
   all_records_list.extend(valid_text)


   
